In [2]:
import pandas as pd
pd.set_option('max_colwidth', 100)
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix
import itertools
import scipy.stats as st
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 500)
import pingouin as pg
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.feature_selection import mutual_info_classif
import random
from random import randint

# Import data

In [15]:
# path where data are saved
path_save_dat_gp_grid1st_full = '../../data/finaldata/dat_gp_grid1st_full.csv'
path_save_dat_en_grid1st_full = '../../data/finaldata/dat_en_grid1st_full.csv'

# load from csv
data_gp = pd.read_csv(path_save_dat_gp_grid1st_full)
data_en = pd.read_csv(path_save_dat_en_grid1st_full)

# Super mild processing

# create subject id column
data_gp = data_gp.rename(columns={'Unnamed: 0': 'Subject'})
data_en = data_en.rename(columns={'Unnamed: 0': 'Subject'})

print(data_gp.shape)
print(data_en.shape)

# combine
data_combined = pd.concat([data_gp, data_en])

print(data_combined.shape)

(497, 801)
(305, 801)
(802, 801)


In [16]:
# check stuff
data_combined.head(3)

,Subject,caseid,weight,sample_type,screener_1ongoing,screener_2impact,screener_3depression,screener_4anxiety,screener_5attention,consent,mood_yn,FNM_Q25_1,FNM_Q25_2,FNM_Q25_3,FNM_Q25_4,FNM_Q25_5,FNM_Q25_6,FNM_Q25_955,FNM_Q25_933,FNM_Q25_open1,mood_years,FNM_Q27_1,FNM_Q27_2,FNM_Q27_3,FNM_Q27_4,FNM_Q27_5,FNM_Q27_6,FNM_Q27_7,FNM_Q27_8,FNM_Q27_9,FNM_Q27_10,FNM_Q27_11,FNM_Q27_12,FNM_Q27_13,FNM_Q27_14,FNM_Q27_15,FNM_Q27_16,FNM_Q27_17,FNM_Q27_18,FNM_Q27_19,FNM_Q27_955,FNM_Q27_933,FNM_Q27_open1,mood_bothered_orig,anxiety_yn,FNM_Q30_m_1,FNM_Q30_m_2,FNM_Q30_m_3,FNM_Q30_m_4,FNM_Q30_m_5,FNM_Q30_m_6,FNM_Q30_m_7,FNM_Q30_m_8,FNM_Q30_m_955,FNM_Q30_m_933,FNM_Q30_open1,anxiety_years,FNM_Q32_1,FNM_Q32_2,FNM_Q32_3,FNM_Q32_4,FNM_Q32_5,FNM_Q32_6,FNM_Q32_7,FNM_Q32_8,FNM_Q32_9,FNM_Q32_10,FNM_Q32_955,FNM_Q32_933,FNM_Q32_open1,anxiety_bothered_orig,attention_yn,FNM_Q35_m_1,FNM_Q35_m_2,FNM_Q35_m_3,FNM_Q35_m_933,FNM_Q35_open1,attention_years,FNM_Q37_m_1,FNM_Q37_m_2,FNM_Q37_m_3,FNM_Q37_m_4,FNM_Q37_m_5,FNM_Q37_m_6,FNM_Q37_m_7,FNM_Q37_m_8,FNM_Q37_m_9,FNM_Q37_m_955,FNM_Q37_m_933,FNM_Q37_open1,attention_bothered_orig,inattention_1,inattention_2,inattention_3,inattention_4,inattention_5,inattention_6,FNM_Q1_attn,inattention_7,inattention_8,inattention_9,hyperactivity_1,hyperactivity_2,hyperactivity_3,hyperactivity_4,hyperactivity_5,impulsivity_1,impulsivity_2,impulsivity_3,impulsivity_4,sct_1,sct_2,sct_3,sct_4,sct_5,sct_6,sct_7,sct_8,sct_9,gad_1,gad_2,gad_3,gad_4,gad_5,gad_6,gad_7,phq_1,phq_2,phq_3,phq_4,phq_5,phq_6,phq_7,phq_8,hitop157,hitop81,hitop34,hitop54,hitop243,hitop182,hitop69,hitop89,hitop50,check_moderately,hitop129,hitop265,hitop124,hitop231,hitop93,hitop67,hitop245,hitop281,hitop141,hitop40,hitop204,hitop21,hitop236,hitop280,hitop84,hitop120,hitop77,hitop92,hitop258,hitop39,hitop254,hitop215,hitop95,hitop106,hitop283,hitop16,hitop20,hitop189,hitop1,hitop136,hitop246,hitop248,hitop257,hitop114,hitop117,hitop250,hitop200,hitop160,hitop23,hitop165,hitop244,hitop9,hitop142,hitop230,hitop149,hitop247,hitop99,hitop66,hitop240,hitop222,hitop90,hitop113,hitop278,hitop203,hitop159,hitop123,hitop275,hitop268,hitop225,hitop143,hitop151,hitop181,hitop211,hitop17,hitop126,hitop5,hitop261,hitop220,check_notatall,hitop15,hitop72,hitop140,hitop109,hitop197,hitop104,todayinattention_1,todayinattention_2,todayinattention_3,todayinattention_4,todayinattention_5,todayinattention_6,todayinattention_7,todayinattention_8,todayinattention_9,todayhyperactivity_1,todayhyperactivity_2,todayhyperactivity_3,todayhyperactivity_4,todayhyperactivity_5,todayimpulsivity_1,todayimpulsivity_2,todayimpulsivity_3,todayimpulsivity_4,todaysct_1,todaysct_2,todaysct_3,todaysct_4,todaysct_5,todaysct_6,todaysct_7,todaysct_8,todaysct_9,today_na1,todaygad_1,todaygad_2,todaygad_3,...,hitop117_recontact,hitop250_recontact,hitop200_recontact,hitop160_recontact,hitop23_recontact,hitop165_recontact,hitop244_recontact,hitop9_recontact,hitop142_recontact,hitop230_recontact,hitop149_recontact,hitop247_recontact,hitop99_recontact,hitop66_recontact,hitop240_recontact,hitop222_recontact,hitop90_recontact,hitop113_recontact,hitop278_recontact,hitop203_recontact,hitop159_recontact,hitop123_recontact,hitop275_recontact,hitop268_recontact,hitop225_recontact,hitop143_recontact,hitop151_recontact,hitop181_recontact,hitop211_recontact,hitop17_recontact,hitop126_recontact,hitop5_recontact,hitop261_recontact,hitop220_recontact,check_notatall_recontact,hitop15_recontact,hitop72_recontact,hitop140_recontact,hitop109_recontact,hitop197_recontact,hitop104_recontact,todayinattention_1_recontact,todayinattention_2_recontact,todayinattention_3_recontact,todayinattention_4_recontact,todayinattention_5_recontact,todayinattention_6_recontact,todayinattention_7_recontact,todayinattention_8_recontact,todayinattention_9_recontact,todayhyperactivity_1_recontact,todayhyperactivity_2_recontact,todayhyperactivity_3_recontact,todayhyperactivity_4_recontact,todayhyperactivity_5_recontact,todayimpulsivity_1_recontact,todayimpulsivi

In [17]:
for c in data_combined.columns:
    if 'baars' in c:
        print(c)

baars_inattention_sum
baars_hyperactivity_sum
baars_impulsivity_sum
baars_sct_sum
baars_inattention_sum_recontact
baars_hyperactivity_sum_recontact
baars_impulsivity_sum_recontact
baars_sct_sum_recontact
baars_sum
baars_sum_recontact


## Replace the OG hitop measures with invariant cores

In [18]:
# define them
for data in data_gp, data_en, data_combined:
    data['hitop_anhedonic_depression_invcore'] = data['hitop39'] + data['hitop77'] + data['hitop84'] + data['hitop93'] + data['hitop182'] + data['hitop230'] + data['hitop246'] 
    data['hitop_anxious_worry_invcore'] = data['hitop34'] + data['hitop89'] + data['hitop203'] + data['hitop248']
    data['hitop_appetite_gain_invcore'] = data['hitop120'] + data['hitop243'] + data['hitop275']
    data['hitop_separation_insecurity_invcore'] = data['hitop50'] + data['hitop69'] + data['hitop81'] + data['hitop136'] + data['hitop151'] + data['hitop197']
    data['hitop_situational_phobia_invcore'] = data['hitop16'] + data['hitop165'] + data['hitop225'] + data['hitop247']
    data['hitop_social_anxiety_invcore'] = data['hitop1'] + data['hitop17'] + data['hitop114'] + data['hitop129'] + data['hitop204'] + data['hitop258']
    data['hitop_well_being_invcore'] = data['hitop9'] + data['hitop23'] + data['hitop149'] + data['hitop200'] + data['hitop244'] + data['hitop250'] + data['hitop281']

# Convergent validity hypotheses

In [19]:
# The p-value helps us determine whether or not we can meaningfully conclude that 
# the population correlation coefficient is different from zero, 
# based on what we observe from the sample.

"We will use these invariant scales in our subsequent tests of convergent and divergent validity. 

For scales in which we cannot establish an invariant core, we will test the relevant hypotheses in just the enriched sample."

In [20]:
def do_correlation(col1, col2):
    corr_result = pg.corr(col1, col2, method = 'spearman')
    my_r = corr_result.iloc[0]['r']
    my_r = "{:.2f}".format(float(my_r))
    my_ci = corr_result.iloc[0]['CI95%']
    my_ci_1 = my_ci[0]
    my_ci_2 = my_ci[1]
    my_p = corr_result.iloc[0]['p-val']
    my_p = "{:.3f}".format(float(my_p)) 
    return(my_p, my_r, my_ci_1, my_ci_2)

def check_convergent_hypotheses_com(mydata=data_combined):
    '''
    tests convergent hypotheses USING COMBINED DATA ONLY, FOR ALL HYPOTHESES
    (for hitop scales we use inv cores when appropriate)
    '''
    assert(visit in ['initial'])
    assert(whichdata in ['genpop', 'enriched', 'combined'])
    print('\nDATA = '+whichdata)    
    if visit == 'initial':
        print('\nVISIT = '+visit)
        # HYPOTHESIS C1: hitop_anhedonic_depression will be positively correlated with PHQ-8
        # Spearman; alpha = .05
        print('\nHYPOTHESIS C1')
        (p, r, ci1, ci2) = do_correlation(mydata.hitop_anhedonic_depression_invcore, mydata.phq_sum)
        if float(p) >= 0.05:
            print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        else:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            if float(r) > 0:
                print('Hyp C1 SUPPORTED: Anhedonic depression (or invariant core for enriched) positively correlates with PHQ-8')
            else:
                print('Hyp C1 NOT SUPPORTED: Anhedonic depression (or invariant core for enriched) DOES NOT positively correlate with PHQ-8')
        
        # HYPOTHESIS C2: hitop_well-being will be negatively correlated with PHQ-8
        # Spearman; alpha = .05
        print('\nHYPOTHESIS C2') 
        if whichdata == 'genpop':
        (p, r, ci1, ci2) = do_correlation(mydata.hitop_well_being_invcore, mydata.phq_sum)
        if float(p) >= 0.05:
            print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        else:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            if float(r) < 0:
                print('Hyp C2 SUPPORTED: Well-being (or invariant core for genpop) negatively correlates with PHQ-8')
            else:
                print('Hyp C2 NOT SUPPORTED: Well-being (or invariant core for genpop) DOES NOT negatively correlate with PHQ-8')
        
        # HYPOTHESIS C3: hitop_anxious_worry will be positively correlated with GAD-7
        # Spearman; alpha = .05
        print('\nHYPOTHESIS C3')              
        (p, r, ci1, ci2) = do_correlation(mydata.hitop_anxious_worry_invcore, mydata.gad_sum)
        if float(p) >= 0.05:
            print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        else:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            if float(r) > 0:
                print('Hyp C3 SUPPORTED: Anxious worry (or invariant core for genpop) positively correlates with GAD-7')
            else:
                print('Hyp C3 NOT SUPPORTED: Anxious worry (or invariant core for genpop) DOES NOT positively correlate with GAD-7') 

        # HYPOTHESIS C4: Higher sum of scores on the HiTOP Appetite Gain and Appetite Loss 
        # will be correlated with responses to PJQ-8 "Poor appetite or overeating"
        print('\nHYPOTHESIS C4')              
        mydata['hitop_appetite_sum'] = mydata['hitop_appetite_gain_invcore'] + mydata['hitop_appetite_loss']
        (p, r, ci1, ci2) = do_correlation(mydata.hitop_appetite_sum, mydata.phq_5)
        if float(p) >= 0.05:
            print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            print('Hyp C4 NOT SUPPORTED: Sum of scores on Appetite Gain and Appetite Loss (or invariant cores) DOES NOT correlate with PHQ item 5 - poor appetite or overeating')         
        else:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            print('Hyp C4 SUPPORTED: Sum of scores on Appetite Gain and Appetite Loss (or invariant cores) is correlated with PHQ item 5 - poor appetite or overeating')

        # HYPOTHESIS C5: the HiTOP Cognitive Problems will be positively correlated with responses 
        # to the PHQ-8 item "Trouble concentrating on things, such as school work, reading or watching television".
        print('\nHYPOTHESIS C5')            
        (p, r, ci1, ci2) = do_correlation(mydata.hitop_cognitive_problems, mydata.phq_7)
        if float(p) >= 0.05:
            print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        else:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            if float(r) > 0:
                print('Hyp C5 SUPPORTED: HiTOP Cognitive problems positive correlates with PHQ-8 item 7 "Trouble concentrating on things..."')
            else:
                print('Hyp C5 NOT SUPPORTED: HiTOP Cognitive problems DOES NOT positive correlate with PHQ-8 item 7 "Trouble concentrating on things..."')         
        
        # HYPOTHESIS C6: The HiTOP Insomnia subscale will be positively correlated with responses 
        # to the PHQ-8 item "Trouble falling or staying asleep, or sleeping too much"
        print('\nHYPOTHESIS C6')            
        (p, r, ci1, ci2) = do_correlation(mydata.hitop_insomnia, mydata.phq_3)        
        if float(p) >= 0.05:
            print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        else:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            if float(r) > 0:
                print('Hyp C6 SUPPORTED: HiTOP Insomnia positive correlates with PHQ-8 item 3 "Trouble falling or staying asleep, or sleeping too much"')
            else:
                print('Hyp C6 NOT SUPPORTED: HiTOP Insomnia DOES NOT positive correlate with PHQ-8 item 3 "Trouble falling or staying asleep, or sleeping too much"')         

        # HYPOTHESIS C7: Higher sum of all HiTOP subscales will be associated with greater likelihood
        # of being bothered by symptoms of a diagnosed mood or anxiety disorder. 
        print('\nHYPOTHESIS C7')              
        mydata['hitop_sum_new'] = mydata['hitop_anhedonic_depression_invcore'] + mydata['hitop_anxious_worry_invcore'] + \
        mydata['hitop_appetite_gain_invcore'] + mydata['hitop_appetite_loss'] + mydata['hitop_cognitive_problems'] + \
        mydata['hitop_hyposomnia'] + mydata['hitop_indecisiveness'] + mydata['hitop_insomnia'] + mydata['hitop_panic'] + \
        mydata['hitop_separation_insecurity_invcore'] + mydata['hitop_shame_guilt'] + mydata['hitop_situational_phobia'] + \
        mydata['hitop_social_anxiety_invcore']
        # using an example from here: https://www.science.smith.edu/~jcrouser/SDS293/labs/lab4-py.html
        formula = 'moodanxiety_bothered ~ hitop_sum_new'        
        model = smf.glm(formula = formula, data=mydata, family=sm.families.Binomial())
        result = model.fit()
        print(result.summary())
        p_val_hitop_sum = format(result.pvalues.iloc[1], 'f')
        if float(p_val_hitop_sum) >= 0.05:
            print(f"\nHitop sum of items is NOT a significant predictor of mood_anxiety_bothered, with p-val = {p_val_hitop_sum}")
            print('Hyp C7 NOT SUPPORTED')         
        else:
            print(f"\nHitop sum of items IS A SIGNIFICANT PREDICTOR of mood_anxiety_bothered, with p-val = {p_val_hitop_sum}")
            print('Hyp C7 SUPPORTED')
            
def check_convergent_hypotheses_en(mydata_com, mydata_gp, mydata_en, visit, whichdata):
    '''
    tests convergent hypotheses USING COMBINED DATA ONLY, FOR ALL HYPOTHESES
    (for hitop scales we use inv cores when appropriate)
    '''
    assert(visit in ['initial'])
    assert(whichdata in ['genpop', 'enriched', 'combined'])
    print('\nDATA = '+whichdata)    
    if visit == 'initial':
        print('\nVISIT = '+visit)
        # HYPOTHESIS C1: hitop_anhedonic_depression will be positively correlated with PHQ-8
        # Spearman; alpha = .05
        print('\nHYPOTHESIS C1')
        if whichdata == 'genpop':
            (p, r, ci1, ci2) = do_correlation(mydata.hitop_anhedonic_depression_invcore, mydata.phq_sum)
        elif whichdata == 'enriched':
            (p, r, ci1, ci2) = do_correlation(mydata.hitop_anhedonic_depression_invcore, mydata.phq_sum) 
        elif whichdata == 'combined':
            (p, r, ci1, ci2) = do_correlation(mydata.hitop_anhedonic_depression_invcore, mydata.phq_sum)
        if float(p) >= 0.05:
            print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        else:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            if float(r) > 0:
                print('Hyp C1 SUPPORTED: Anhedonic depression (or invariant core for enriched) positively correlates with PHQ-8')
            else:
                print('Hyp C1 NOT SUPPORTED: Anhedonic depression (or invariant core for enriched) DOES NOT positively correlate with PHQ-8')
        
        # HYPOTHESIS C2: hitop_well-being will be negatively correlated with PHQ-8
        # Spearman; alpha = .05
        print('\nHYPOTHESIS C2') 
        if whichdata == 'genpop':
            (p, r, ci1, ci2) = do_correlation(mydata.hitop_well_being_invcore, mydata.phq_sum)
        elif whichdata == 'enriched':
            (p, r, ci1, ci2) = do_correlation(mydata.hitop_well_being_invcore, mydata.phq_sum)      
        elif whichdata == 'combined':
            (p, r, ci1, ci2) = do_correlation(mydata.hitop_well_being_invcore, mydata.phq_sum)
        if float(p) >= 0.05:
            print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        else:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            if float(r) < 0:
                print('Hyp C2 SUPPORTED: Well-being (or invariant core for genpop) negatively correlates with PHQ-8')
            else:
                print('Hyp C2 NOT SUPPORTED: Well-being (or invariant core for genpop) DOES NOT negatively correlate with PHQ-8')
        
        # HYPOTHESIS C3: hitop_anxious_worry will be positively correlated with GAD-7
        # Spearman; alpha = .05
        print('\nHYPOTHESIS C3')            
        if whichdata == 'genpop':
            (p, r, ci1, ci2) = do_correlation(mydata.hitop_anxious_worry_invcore, mydata.gad_sum)
        elif whichdata == 'enriched':   
            (p, r, ci1, ci2) = do_correlation(mydata.hitop_anxious_worry_invcore, mydata.gad_sum)
        elif whichdata == 'combined':   
            (p, r, ci1, ci2) = do_correlation(mydata.hitop_anxious_worry_invcore, mydata.gad_sum)
        if float(p) >= 0.05:
            print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        else:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            if float(r) > 0:
                print('Hyp C3 SUPPORTED: Anxious worry (or invariant core for genpop) positively correlates with GAD-7')
            else:
                print('Hyp C3 NOT SUPPORTED: Anxious worry (or invariant core for genpop) DOES NOT positively correlate with GAD-7') 

        # HYPOTHESIS C4: Higher sum of scores on the HiTOP Appetite Gain and Appetite Loss 
        # will be correlated with responses to PJQ-8 "Poor appetite or overeating"
        print('\nHYPOTHESIS C4')            
        if whichdata == 'genpop':
            mydata['hitop_appetite_sum'] = mydata['hitop_appetite_gain_invcore'] + mydata['hitop_appetite_loss']
        elif whichdata == 'enriched':   
            mydata['hitop_appetite_sum'] = mydata['hitop_appetite_gain_invcore'] + mydata['hitop_appetite_loss']
        elif whichdata == 'combined':   
            mydata['hitop_appetite_sum'] = mydata['hitop_appetite_gain_invcore'] + mydata['hitop_appetite_loss']
        (p, r, ci1, ci2) = do_correlation(mydata.hitop_appetite_sum, mydata.phq_5)
        if float(p) >= 0.05:
            print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            print('Hyp C4 NOT SUPPORTED: Sum of scores on Appetite Gain and Appetite Loss (or invariant cores) DOES NOT correlate with PHQ item 5 - poor appetite or overeating')         
        else:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            print('Hyp C4 SUPPORTED: Sum of scores on Appetite Gain and Appetite Loss (or invariant cores) is correlated with PHQ item 5 - poor appetite or overeating')

        # HYPOTHESIS C5: the HiTOP Cognitive Problems will be positively correlated with responses 
        # to the PHQ-8 item "Trouble concentrating on things, such as school work, reading or watching television".
        print('\nHYPOTHESIS C5')            
        (p, r, ci1, ci2) = do_correlation(mydata.hitop_cognitive_problems, mydata.phq_7)
        if float(p) >= 0.05:
            print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        else:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            if float(r) > 0:
                print('Hyp C5 SUPPORTED: HiTOP Cognitive problems positive correlates with PHQ-8 item 7 "Trouble concentrating on things..."')
            else:
                print('Hyp C5 NOT SUPPORTED: HiTOP Cognitive problems DOES NOT positive correlate with PHQ-8 item 7 "Trouble concentrating on things..."')         
        
        # HYPOTHESIS C6: The HiTOP Insomnia subscale will be positively correlated with responses 
        # to the PHQ-8 item "Trouble falling or staying asleep, or sleeping too much"
        print('\nHYPOTHESIS C6')            
        (p, r, ci1, ci2) = do_correlation(mydata.hitop_insomnia, mydata.phq_3)        
        if float(p) >= 0.05:
            print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        else:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            if float(r) > 0:
                print('Hyp C6 SUPPORTED: HiTOP Insomnia positive correlates with PHQ-8 item 3 "Trouble falling or staying asleep, or sleeping too much"')
            else:
                print('Hyp C6 NOT SUPPORTED: HiTOP Insomnia DOES NOT positive correlate with PHQ-8 item 3 "Trouble falling or staying asleep, or sleeping too much"')         

        # HYPOTHESIS C7: Higher sum of all HiTOP subscales will be associated with greater likelihood
        # of being bothered by symptoms of a diagnosed mood or anxiety disorder. 
        print('\nHYPOTHESIS C7')            
        if whichdata == 'genpop':
            mydata['hitop_sum_new'] = mydata['hitop_anhedonic_depression_invcore'] + mydata['hitop_anxious_worry_invcore'] + \
            mydata['hitop_appetite_gain_invcore'] + mydata['hitop_appetite_loss'] + mydata['hitop_cognitive_problems'] + \
            mydata['hitop_hyposomnia'] + mydata['hitop_indecisiveness'] + mydata['hitop_insomnia'] + mydata['hitop_panic'] + \
            mydata['hitop_separation_insecurity_invcore'] + mydata['hitop_shame_guilt'] + mydata['hitop_situational_phobia'] + \
            mydata['hitop_social_anxiety_invcore']
        elif whichdata == 'enriched':   
            mydata['hitop_sum_new'] = mydata['hitop_anhedonic_depression_invcore'] + mydata['hitop_anxious_worry_invcore'] + \
            mydata['hitop_appetite_gain_invcore'] + mydata['hitop_appetite_loss'] + mydata['hitop_cognitive_problems'] + \
            mydata['hitop_hyposomnia'] + mydata['hitop_indecisiveness'] + mydata['hitop_insomnia'] + mydata['hitop_panic'] + \
            mydata['hitop_separation_insecurity_invcore'] + mydata['hitop_shame_guilt'] + mydata['hitop_situational_phobia'] + \
            mydata['hitop_social_anxiety_invcore']
        elif whichdata == 'combined':   
            mydata['hitop_sum_new'] = mydata['hitop_anhedonic_depression_invcore'] + mydata['hitop_anxious_worry_invcore'] + \
            mydata['hitop_appetite_gain_invcore'] + mydata['hitop_appetite_loss'] + mydata['hitop_cognitive_problems'] + \
            mydata['hitop_hyposomnia'] + mydata['hitop_indecisiveness'] + mydata['hitop_insomnia'] + mydata['hitop_panic'] + \
            mydata['hitop_separation_insecurity_invcore'] + mydata['hitop_shame_guilt'] + mydata['hitop_situational_phobia'] + \
            mydata['hitop_social_anxiety_invcore']
        # using an example from here: https://www.science.smith.edu/~jcrouser/SDS293/labs/lab4-py.html
        formula = 'moodanxiety_bothered ~ hitop_sum_new'        
        model = smf.glm(formula = formula, data=mydata, family=sm.families.Binomial())
        result = model.fit()
        print(result.summary())
        p_val_hitop_sum = format(result.pvalues.iloc[1], 'f')
        if float(p_val_hitop_sum) >= 0.05:
            print(f"\nHitop sum of items is NOT a significant predictor of mood_anxiety_bothered, with p-val = {p_val_hitop_sum}")
            print('Hyp C7 NOT SUPPORTED')         
        else:
            print(f"\nHitop sum of items IS A SIGNIFICANT PREDICTOR of mood_anxiety_bothered, with p-val = {p_val_hitop_sum}")
            print('Hyp C7 SUPPORTED')

In [21]:
check_convergent_hypotheses(visit = 'initial', whichdata = 'combined')


DATA = combined

VISIT = initial

HYPOTHESIS C1
Correlation Significant with p-val = 0.000 and r = 0.87(0.85, 0.89)
Hyp C1 SUPPORTED: Anhedonic depression (or invariant core for enriched) positively correlates with PHQ-8

HYPOTHESIS C2
Correlation Significant with p-val = 0.000 and r = -0.54(-0.59, -0.49)
Hyp C2 SUPPORTED: Well-being (or invariant core for genpop) negatively correlates with PHQ-8

HYPOTHESIS C3
Correlation Significant with p-val = 0.000 and r = 0.89(0.87, 0.9)
Hyp C3 SUPPORTED: Anxious worry (or invariant core for genpop) positively correlates with GAD-7

HYPOTHESIS C4
Correlation Significant with p-val = 0.000 and r = 0.77(0.74, 0.8)
Hyp C4 SUPPORTED: Sum of scores on Appetite Gain and Appetite Loss (or invariant cores) is correlated with PHQ item 5 - poor appetite or overeating

HYPOTHESIS C5
Correlation Significant with p-val = 0.000 and r = 0.81(0.79, 0.83)
Hyp C5 SUPPORTED: HiTOP Cognitive problems positive correlates with PHQ-8 item 7 "Trouble concentrating on t

# Divergent validity hypotheses

In [22]:
whichdata = 'combined'
visit = 'initial'

print('\nDATA = ' + whichdata)    
print('\nVISIT = ' + visit)

# PREP FOR HYP D1 AND HYP D2

# -----------------------------------------------------------------------
# D1: There will be a stronger correlation between the sum of HiTOP scales and the sum of GAD-7 and PHQ-8 scores 
#     than with BAARS-IV total score

mydata['gad_phq_sum'] = mydata['gad_sum'] + mydata['phq_sum']
mydata['hitop_sum_invcores'] = mydata['hitop_anhedonic_depression_invcore'] + mydata['hitop_anxious_worry_invcore'] + \
            mydata['hitop_appetite_gain_invcore'] + mydata['hitop_appetite_loss'] + mydata['hitop_cognitive_problems'] + \
            mydata['hitop_hyposomnia'] + mydata['hitop_indecisiveness'] + mydata['hitop_insomnia'] + mydata['hitop_panic'] + \
            mydata['hitop_separation_insecurity_invcore'] + mydata['hitop_shame_guilt'] + mydata['hitop_situational_phobia'] + \
            mydata['hitop_social_anxiety_invcore']
(p_hitop_gadphq, r_hitop_gadphq, ci1_hitop_gadphq, ci2_hitop_gadphq) = do_correlation(stats.zscore(mydata.hitop_sum_invcores), stats.zscore(mydata.gad_phq_sum))
(p_hitop_baars, r_hitop_baars, ci1_hitop_baars, ci2_hitop_baars) = do_correlation(stats.zscore(mydata.hitop_sum_invcores), stats.zscore(mydata.baars_sum))
corr_result_orig = float(r_hitop_gadphq) - float(r_hitop_baars)

# -----------------------------------------------------------------------
# D2: The sum of HiTOP scales will be a better predictor of being bothered by symptoms of a diagnosed anxiety disorder 
# than of being bothered by symptoms of ADHD.

X_hitop_sum_invcores = []
for x in mydata['hitop_sum_invcores'].tolist():
    X_hitop_sum_invcores.append([x])
    
# https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_classif.html
mi_hitop_moodanxboth = mutual_info_classif(X_hitop_sum_invcores, mydata['moodanxiety_bothered'].tolist())
mi_hitop_attnboth = mutual_info_classif(X_hitop_sum_invcores, mydata['attention_bothered'].tolist())
MI_diff_orig_D2 = mi_hitop_moodanxboth - mi_hitop_attnboth

# -----------------------------------------------------------------------
# D3: The BAARS-IV total score will be a better predictor of being bothered by symptoms of ADHD 
# than of a diagnosed mood or anxiety disorder.

X_baars_sum = []
for x in mydata['baars_sum'].tolist():
    X_baars_sum.append([x])
    
mi_baars_attnboth = mutual_info_classif(X_baars_sum, mydata['attention_bothered'].tolist())
mi_baars_moodanxboth = mutual_info_classif(X_baars_sum, mydata['moodanxiety_bothered'].tolist())
MI_diff_orig_D3 = mi_baars_attnboth - mi_baars_moodanxboth

# -----------------------------------------------------------------------
# D4: The sum of HiTOP mood disorder scales will be a better predictor of being bothered by symptoms 
# of a diagnosed mood disorder than of being bothered by symptoms of a diagnosed anxiety disorder.

X_hitop_depression = []
for x in mydata['hitop_anhedonic_depression_invcore'].tolist():
    X_hitop_depression.append([x])

mi_hitopdepr_moodboth = mutual_info_classif(X_hitop_depression, mydata['mood_bothered'].tolist())
mi_hitopdepr_anxboth = mutual_info_classif(X_hitop_depression, mydata['anxiety_bothered'].tolist())
MI_diff_orig_D4 = mi_hitopdepr_moodboth - mi_hitopdepr_anxboth

# -----------------------------------------------------------------------
# D5: The sum of HiTOP anxiety disorder scales will be a better predictor of being bothered by symptoms 
# of a diagnosed anxiety disorder than of being bothered by symptoms of a diagnosed mood disorder.

X_hitop_anxiety = []
for x in mydata['hitop_anxious_worry_invcore'].tolist():
    X_hitop_anxiety.append([x])
    
mi_hitopanxiety_anxboth = mutual_info_classif(X_hitop_anxiety, mydata['anxiety_bothered'].tolist())
mi_hitopanxiety_moodboth = mutual_info_classif(X_hitop_anxiety, mydata['mood_bothered'].tolist())
MI_diff_orig_D5 = mi_hitopanxiety_anxboth - mi_hitopanxiety_moodboth

# -----------------------------------------------------------------------
corr_result_perm = []
MI_diff_perm_D2 = []
MI_diff_perm_D3 = []
MI_diff_perm_D4 = []
MI_diff_perm_D5 = []

# permutations
for permut in range (1000):
    rand_seed = randint(1, 10000)
    dat_shuffled = pd.DataFrame()
    dat_shuffled['hitop_sum_invcores'] = mydata['hitop_sum_invcores']
    np.random.seed(rand_seed)
    dat_shuffled['baars_sum'] = np.random.permutation(mydata['baars_sum'])
    np.random.seed(rand_seed)
    dat_shuffled['gad_phq_sum'] = np.random.permutation(mydata['gad_phq_sum'])
    np.random.seed(rand_seed)
    dat_shuffled['moodanxiety_bothered'] = np.random.permutation(mydata['moodanxiety_bothered'])
    np.random.seed(rand_seed)
    dat_shuffled['attention_bothered'] = np.random.permutation(mydata['attention_bothered'])
    np.random.seed(rand_seed)
    dat_shuffled['mood_bothered'] = np.random.permutation(mydata['mood_bothered'])
    np.random.seed(rand_seed)
    dat_shuffled['anxiety_bothered'] = np.random.permutation(mydata['anxiety_bothered'])
    # D1
    (p_hitop_gadphq_curr, r_hitop_gadphq_curr, ci1_hitop_gadphq_curr, ci2_hitop_gadphq_curr) = do_correlation(stats.zscore(dat_shuffled.hitop_sum_invcores), stats.zscore(dat_shuffled.gad_phq_sum))
    (p_hitop_baars_curr, r_hitop_baars_curr, ci1_hitop_baars_curr, ci2_hitop_baars_curr) = do_correlation(stats.zscore(dat_shuffled.hitop_sum_invcores), stats.zscore(dat_shuffled.baars_sum))
    corr_result_perm.append(float(r_hitop_gadphq_curr) - float(r_hitop_baars_curr))
    # D2
    mi_hitop_moodanxboth_perm = mutual_info_classif(X_hitop_sum_invcores, dat_shuffled['moodanxiety_bothered'].tolist())
    mi_hitop_attnboth_perm = mutual_info_classif(X_hitop_sum_invcores, dat_shuffled['attention_bothered'].tolist())   
    MI_diff_perm_D2.append(mi_hitop_moodanxboth_perm - mi_hitop_attnboth_perm)
    # D3
    mi_baars_attnboth_perm = mutual_info_classif(X_baars_sum, dat_shuffled['attention_bothered'].tolist())
    mi_baars_moodanxboth_perm = mutual_info_classif(X_baars_sum, dat_shuffled['moodanxiety_bothered'].tolist())
    MI_diff_perm_D3.append(mi_baars_attnboth_perm - mi_baars_moodanxboth_perm)
    # D4
    mi_hitopdepr_moodboth_perm = mutual_info_classif(X_hitop_depression, dat_shuffled['mood_bothered'].tolist())
    mi_hitopdepr_anxboth_perm = mutual_info_classif(X_hitop_depression, dat_shuffled['anxiety_bothered'].tolist())
    MI_diff_perm_D4.append(mi_hitopdepr_moodboth_perm - mi_hitopdepr_anxboth_perm)    
    # D5
    mi_hitopanxiety_anxboth_perm = mutual_info_classif(X_hitop_anxiety, dat_shuffled['anxiety_bothered'].tolist())
    mi_hitopanxiety_moodboth_perm = mutual_info_classif(X_hitop_anxiety, dat_shuffled['mood_bothered'].tolist())
    MI_diff_perm_D5.append(mi_hitopanxiety_anxboth_perm - mi_hitopanxiety_moodboth_perm)

# -----------------------------------------------------------------------
# D1: There will be a stronger correlation between the sum of HiTOP scales and the sum of GAD-7 and PHQ-8 scores 
#     than with BAARS-IV total score
print('\nHYPOTHESIS D1')

# Perform one-sample t-test
t_stat, p_value = stats.ttest_1samp(corr_result_perm, corr_result_orig)
print("\nT statistic:", t_stat)
print("P-value:", p_value)

# Setting significance level
alpha = 0.05

# Interpret the results
if p_value < alpha:
    print("\nHyp D1: Reject the null hypothesis; there is a significant difference between the permutation mean and the actual mean.")
else:
    print("\nHyp D1: Fail to reject the null hypothesis; there is no significant difference between the permutation mean and the actual mean.")

# -----------------------------------------------------------------------
# D2: The sum of HiTOP scales will be a better predictor of being bothered by symptoms of a diagnosed anxiety disorder 
# than of being bothered by symptoms of ADHD.
print('\nHYPOTHESIS D2')

print('\nActual MI between hitop and mood_anx_bothered: ' + str(mi_hitop_moodanxboth))
print('Actual MI between hitop and attnt_bothered: ' + str(mi_hitop_attnboth))

# Perform one-sample t-test
t_stat, p_value = stats.ttest_1samp(MI_diff_perm_D2, MI_diff_orig_D2)
print("\nT statistic:", t_stat)
print("P-value:", p_value)

# Setting significance level
alpha = 0.05

# Interpret the results
if p_value < alpha:
    print("\nHyp D2: Reject the null hypothesis; there is a significant difference between the permutation mean and the actual mean.")
else:
    print("\nHyp D2: Fail to reject the null hypothesis; there is no significant difference between the permutation mean and the actual mean.")

# -----------------------------------------------------------------------
# D3: The BAARS-IV total score will be a better predictor of being bothered by symptoms of ADHD 
# than of a diagnosed mood or anxiety disorder.
print('\nHYPOTHESIS D3')

print('\nActual MI between baars and mood_anx_bothered: ' + str(mi_baars_moodanxboth))
print('Actual MI between baars and attnt_bothered: ' + str(mi_baars_attnboth))

# Perform one-sample t-test
t_stat, p_value = stats.ttest_1samp(MI_diff_perm_D3, MI_diff_orig_D3)
print("\nT statistic:", t_stat)
print("P-value:", p_value)

# Setting significance level
alpha = 0.05

# Interpret the results
if p_value < alpha:
    print("\nHyp D3: Reject the null hypothesis; there is a significant difference between the permutation mean and the actual mean.")
else:
    print("\nHyp D3: Fail to reject the null hypothesis; there is no significant difference between the permutation mean and the actual mean.")

# -----------------------------------------------------------------------
# D4: The sum of HiTOP mood disorder scales will be a better predictor of being bothered by symptoms 
# of a diagnosed mood disorder than of being bothered by symptoms of a diagnosed anxiety disorder.
print('\nHYPOTHESIS D4:')

print('\nActual MI between HiTOP depresson scale and and mood_bothered: ' + str(mi_hitopdepr_moodboth))
print('Actual MI between HiTOP depression scale and anxiety_bothered: ' + str(mi_hitopdepr_anxboth))

# Perform one-sample t-test
t_stat, p_value = stats.ttest_1samp(MI_diff_perm_D4, MI_diff_orig_D4)
print("\nT statistic:", t_stat)
print("P-value:", p_value)

# Setting significance level
alpha = 0.05

# Interpret the results
if p_value < alpha:
    print("\nHyp D4: Reject the null hypothesis; there is a significant difference between the permutation mean and the actual mean.")
else:
    print("\nHyp D4: Fail to reject the null hypothesis; there is no significant difference between the permutation mean and the actual mean.")

# -----------------------------------------------------------------------
# D5: The sum of HiTOP anxiety disorder scales will be a better predictor of being bothered by symptoms 
# of a diagnosed anxiety disorder than of being bothered by symptoms of a diagnosed mood disorder.
print('\nHYPOTHESIS D5')

X_hitop_anxiety = []
for x in mydata['hitop_anxious_worry_invcore'].tolist():
    X_hitop_anxiety.append([x])
    
mi_hitopanxiety_anxboth = mutual_info_classif(X_hitop_anxiety, mydata['anxiety_bothered'].tolist())
mi_hitopanxiety_moodboth = mutual_info_classif(X_hitop_anxiety, mydata['mood_bothered'].tolist())
MI_diff_orig_D5 = mi_hitopanxiety_anxboth - mi_hitopanxiety_moodboth

print('\nActual MI between HiTOP anxiety scale and and anxiety_bothered: ' + str(mi_hitopanxiety_anxboth))
print('Actual MI between HiTOP anxiety scale and mood_bothered: ' + str(mi_hitopanxiety_moodboth))

# Perform one-sample t-test
t_stat, p_value = stats.ttest_1samp(MI_diff_perm_D5, MI_diff_orig_D5)
print("\nT statistic:", t_stat)
print("P-value:", p_value)

# Setting significance level
alpha = 0.05

# Interpret the results
if p_value < alpha:
    print("\nHyp D5: Reject the null hypothesis; there is a significant difference between the permutation mean and the actual mean.")
else:
    print("\nHyp D5: Fail to reject the null hypothesis; there is no significant difference between the permutation mean and the actual mean.")



DATA = combined

VISIT = initial

HYPOTHESIS D1

T statistic: -139.60105944213942
P-value: 0.0

Hyp D1: Reject the null hypothesis; there is a significant difference between the permutation mean and the actual mean.

HYPOTHESIS D2

Actual MI between hitop and mood_anx_bothered: [0.22808654]
Actual MI between hitop and attnt_bothered: [0.06820378]

T statistic: [-385.06636099]
P-value: [0.]

Hyp D2: Reject the null hypothesis; there is a significant difference between the permutation mean and the actual mean.

HYPOTHESIS D3

Actual MI between baars and mood_anx_bothered: [0.18468274]
Actual MI between baars and attnt_bothered: [0.10911692]

T statistic: [186.75994623]
P-value: [0.]

Hyp D3: Reject the null hypothesis; there is a significant difference between the permutation mean and the actual mean.

HYPOTHESIS D4:

Actual MI between HiTOP depresson scale and and mood_bothered: [0.21784478]
Actual MI between HiTOP depression scale and anxiety_bothered: [0.16856773]

T statistic: [-111